In [1]:
import os
import sys
import io

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, '.')
from models import MLP

os.makedirs('gifs', exist_ok=True)

In [ ]:
LABEL_MAP = {'red': 0, 'blue': 1}
COLORS = {0: 'red', 1: 'blue'}

def make_combined_frame(models, neuron_counts, X_np, y_np, epoch):
    fig, axes = plt.subplots(2, 5, figsize=(22, 8))
    axes = axes.flatten()

    margin = 0.5
    x_min, x_max = X_np[:, 0].min() - margin, X_np[:, 0].max() + margin
    y_min, y_max = X_np[:, 1].min() - margin, X_np[:, 1].max() + margin

    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    X_t = torch.tensor(X_np)
    y_t = torch.tensor(y_np)
    point_colors = [COLORS[int(l)] for l in y_np]

    for ax, model, n in zip(axes, models, neuron_counts):
        model.eval()
        with torch.no_grad():
            Z = torch.softmax(model(grid), dim=1)[:, 1].numpy().reshape(xx.shape)
            acc = (model(X_t).argmax(dim=1) == y_t).float().mean().item()

        ax.contourf(xx, yy, Z, levels=[-0.001, 0.5, 1.001], colors=['#FF4444', '#4444FF'], alpha=0.35)
        if Z.min() < 0.5 < Z.max():  # only draw boundary when both regions exist
            ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=0.8)
        ax.scatter(X_np[:, 0], X_np[:, 1], c=point_colors,
                   edgecolors='k', linewidths=0.4, s=18, zorder=5)
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f'{n} neuron{"s" if n > 1 else ""}  (acc={acc:.2f})', fontsize=10)

    fig.suptitle(f'Epoch {epoch}', fontsize=14)
    fig.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=80, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).copy()

In [ ]:
DATASET_PATHS = [
    'datasets/dataset1_two_circles.csv',
    'datasets/dataset2_three_circles.csv',
    'datasets/dataset3_stripes.csv',
    'datasets/dataset4_sine.csv',
    'datasets/dataset5_moons.csv',
]
NEURON_COUNTS = list(range(1, 11))  # 1 through 10

EPOCHS = 1000
LR = 0.1
CAPTURE_EVERY = 10   # one GIF frame per N epochs
GIF_DURATION = 150   # ms per frame

for path in DATASET_PATHS:
    name = os.path.splitext(os.path.basename(path))[0]
    print(f"\n=== {name} ===")

    df = pd.read_csv(path)
    X_np = df[['x', 'y']].values.astype(np.float32)
    y_np = df['label'].map(LABEL_MAP).values.astype(np.int64)
    X = torch.tensor(X_np)
    y = torch.tensor(y_np)

    models = [MLP(hidden_dims=[n]) for n in NEURON_COUNTS]
    optimizers = [torch.optim.SGD(m.parameters(), lr=LR) for m in models]
    criterion = nn.CrossEntropyLoss()

    frames = []

    for epoch in range(1, EPOCHS + 1):
        for model, opt in zip(models, optimizers):
            model.train()
            opt.zero_grad()
            criterion(model(X), y).backward()
            opt.step()

        if epoch == 1 or epoch % CAPTURE_EVERY == 0:
            frames.append(make_combined_frame(models, NEURON_COUNTS, X_np, y_np, epoch))

        if epoch % 200 == 0:
            print(f"  epoch {epoch}/{EPOCHS}")

    gif_path = f'gifs/{name}.gif'
    frames[0].save(
        gif_path,
        save_all=True,
        append_images=frames[1:],
        loop=0,
        duration=GIF_DURATION,
    )
    print(f"  saved -> {gif_path}")